# SRUL: PathMNIST and CelebA-64 knob + CFG sweeps

This notebook trains one RFM prior per `sigma_enc` and evaluates each prior at CFG scales 1.0, 1.5, and 2.0.


In [ ]:
!pip install -q "torchmetrics[image]" torch-fidelity lpips medmnist datasets huggingface_hub pandas matplotlib
from google.colab import drive
drive.mount("/content/drive")


Clone or upload this repository, then set the repository and checkpoint paths in the next cell. Trained checkpoints are not included in the repository.

In [ ]:
from pathlib import Path

REPO = Path("/content/srul-generative-modeling")
PATHMNIST_AE = Path("/content/drive/MyDrive/path/to/pathmnist/autoencoder_final.pt")
CELEBA_AE = Path("/content/drive/MyDrive/path/to/celeba/autoencoder_final.pt")
CELEBA_CACHE = Path("/content/drive/MyDrive/path/to/celeba_cache")
OUTPUT_ROOT = Path("/content/drive/MyDrive/SRUL_Final_Comparisons")

for name, path in {
    "repository": REPO,
    "PathMNIST checkpoint": PATHMNIST_AE,
    "CelebA checkpoint": CELEBA_AE,
}.items():
    print(f"{name}: {path} | exists={path.exists()}")

In [ ]:
# Update the three paths above before starting the sweeps.

## PathMNIST


In [ ]:
!python {REPO}/src/srul_cross_dataset_knob_cfg_sweep.py \
  --dataset pathmnist \
  --data-root "/content/data" \
  --out-dir "{OUTPUT_ROOT}/PathMNIST_knob_cfg" \
  --seed 0 --image-size 32 --num-classes 9 \
  --train-samples 0 --test-samples 7180 \
  --batch-size 128 --metric-batch-size 128 --num-workers 2 \
  --base-channels 96 --latent-channels 32 \
  --ae-checkpoint "{PATHMNIST_AE}" \
  --sigma-values 0.05 0.15 0.30 \
  --guidance-scales 1.0 1.5 2.0 \
  --prior-epochs 80 --prior-lr 2e-4 --prior-width 256 --prior-depth 6 \
  --time-dim 128 --time-sampling logit_normal \
  --label-drop-prob 0.10 --ema-decay 0.999 \
  --sample-steps 100 --metric-samples 7180 --pr-samples 5000 \
  --recon-metric-samples 5000 --pr-chunk-size 256 --pr-nearest-k 5 \
  --checkpoint-every 5 --amp --resume

## CelebA-64


In [ ]:
!python {REPO}/src/srul_cross_dataset_knob_cfg_sweep.py \
  --dataset celeba64 \
  --data-root "{CELEBA_CACHE}" \
  --out-dir "{OUTPUT_ROOT}/CelebA64_knob_cfg" \
  --seed 0 --image-size 64 --num-classes 2 \
  --train-samples 30000 --test-samples 5000 \
  --batch-size 64 --metric-batch-size 64 --num-workers 2 \
  --base-channels 64 --latent-channels 32 \
  --ae-checkpoint "{CELEBA_AE}" \
  --sigma-values 0.05 0.15 0.30 \
  --guidance-scales 1.0 1.5 2.0 \
  --prior-epochs 60 --prior-lr 2e-4 --prior-width 256 --prior-depth 6 \
  --time-dim 128 --time-sampling logit_normal \
  --label-drop-prob 0.10 --ema-decay 0.999 \
  --sample-steps 100 --metric-samples 5000 --pr-samples 5000 \
  --recon-metric-samples 5000 --pr-chunk-size 256 --pr-nearest-k 5 \
  --checkpoint-every 5 --hf-dataset "flwrlabs/celeba" \
  --celeba-attribute "Smiling" --hf-shuffle-buffer 10000 \
  --amp --resume

## Combine and inspect results


In [ ]:
!python {REPO}/src/collect_cross_dataset_results.py \
  --pathmnist-csv "{OUTPUT_ROOT}/PathMNIST_knob_cfg/seed_0/knob_cfg_sweep_metrics.csv" \
  --celeba-csv "{OUTPUT_ROOT}/CelebA64_knob_cfg/seed_0/knob_cfg_sweep_metrics.csv" \
  --output "{OUTPUT_ROOT}/cross_dataset_knob_cfg_results.csv"